# OVENTI · Calibre — Electricity / ECL application

**One-step-ahead hourly forecasting and 90% prediction intervals.**

0. Packages and Oventi styling.
1. Data and quality checks; client displays limited to the first five clients.
2. Chronology, ACF/PACF, SARIMA order selection and five repository forecasters.
3. EnbPI, ACI, Conformal PID; pair diagnostics and five-metric comparisons by forecaster.

**13 pairs**: SARIMA + native Gaussian interval; moving average, MSTL decomposition,
CatBoost and OGD, each with three calibrators. CatBoost is **gradient-boosted
trees, not a neural network**. Forecasting and calibration use `calibre`;
there is no dependency on `conformal-ts-bench`.

**Prediction intervals (PIs)** concern a future observation; the final **95% CIs**
concern mean performance. The data are the LSTNet ECL matrix, with no source
timestamps or verifiable physical units. Client identifiers are positional.
For setup instructions, see [`README.md`](README.md).

## 0) Imports, configuration and Oventi styling

In [ ]:
import importlib.metadata
import json
import platform
import sys
from dataclasses import asdict
from datetime import UTC, datetime
from pathlib import Path

import matplotlib.pyplot as plt  # ty: ignore[unresolved-import]
import numpy as np
import pandas as pd
import seaborn as sns  # ty: ignore[unresolved-import]
from IPython.display import display
from matplotlib.colors import LinearSegmentedColormap  # ty: ignore[unresolved-import]
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "src/calibre").is_dir() and (p / "pyproject.toml").exists()
)
NOTEBOOK_DIR = ROOT / "notebooks"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from electricity_application import (  # ty: ignore[unresolved-import]  # noqa: E402
    OVENTI,
    SOURCE_SHA256,
    SOURCE_URL,
    Config,
    ExperimentReporter,
    chronological_splits,
    describe_data,
    fold_metrics,
    identify_sarima,
    load_electricity,
    run_family,
    summarize,
)

import calibre  # noqa: E402

assert Path(calibre.__file__).resolve().is_relative_to(ROOT / "src"), (
    "Use the local Calibre installation: launch with uv run from the repository."
)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 85)
print("Repository:", ROOT)
print("Calibre:", calibre.__file__)

### Parameters fixed before evaluation

Five contiguous test blocks: 30 training days, 7 calibration days and 7 test days.
Each block contains $H=168$ targets, but every forecast has **lead $h=1$**.
The model is reset at the beginning of each fold. The target, SARIMA grid,
controller gains and illustrated fold are fixed without inspecting test errors.
The final bootstrap resamples **blocks of folds**, not independent hours.

In [ ]:
TARGET_COLUMN = 0
CONFIG = Config(
    n_split=5,
    train_size=30 * 24,
    calibration_size=7 * 24,
    test_size=7 * 24,
    level=0.90,
    n_lags=24,
    period=24,
    moving_window=24,
    sarima_d=1,
    sarima_method="CSS-ML",
    seasonal_order=(1, 0, 0),
    catboost_iterations=200,
    catboost_depth=5,
    ogd_eta=0.001,
    ogd_l2=0.001,
    aci_gamma=0.005,
    pid_rate=0.05,
    pid_integral=0.10,
    seed=42,
)
SARIMA_MAX_P, SARIMA_MAX_Q = 2, 2
PLOT_FOLD = 0
ROLLING_COVERAGE_WINDOW = 24
CONFIDENCE_METRICS = 0.95
N_BOOTSTRAP_CI = 2000
BOOTSTRAP_BLOCK_LENGTH = 2
WRITE_RESULTS = True

assert 0 <= PLOT_FOLD < CONFIG.n_split
assert CONFIG.test_size >= ROLLING_COVERAGE_WINDOW
CREATED_AT = datetime.now(UTC)
OUTPUT = ROOT / "results/electricity_notebook" / CREATED_AT.strftime("%Y%m%dT%H%M%S-%fZ")
if WRITE_RESULTS:
    OUTPUT.mkdir(parents=True, exist_ok=False)
reporter = ExperimentReporter(
    level=CONFIG.level,
    plot_fold=PLOT_FOLD,
    rolling_window=ROLLING_COVERAGE_WINDOW,
    output=OUTPUT if WRITE_RESULTS else None,
)
display(pd.Series(asdict(CONFIG), name="Configuration"))

In [ ]:
sns.set_theme(style="whitegrid", context="notebook", palette=list(OVENTI.values()))
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "savefig.dpi": 170,
        "axes.titleweight": "bold",
        "axes.titlecolor": OVENTI["medium"],
        "axes.labelcolor": OVENTI["dark"],
        "axes.edgecolor": OVENTI["medium"],
        "font.size": 10,
    }
)
# No reconstructed logo. Supply the original JPEG to this location if desired.
LOGO = NOTEBOOK_DIR / "assets/OVENTI-logo.jpeg"
if LOGO.exists():
    from IPython.display import Image

    display(Image(filename=str(LOGO), width=160))
else:
    print("Original OVENTI logo unavailable; exact Oventi palette is applied.")

## I) Data download and descriptive statistics

Public source **pinned to a commit**, with SHA-256 verification on every read.
The complete file contains **26,304 hourly rows × 321 anonymous clients**.
It is downloaded on the first run to `data/electricity_notebook/`, then cached locally.
No dates are invented in the analysis; no rows are removed or imputed.
Descriptive statistics are computed for the entire file, **without selecting variables
or hyperparameters**, but **only the first five clients are displayed** in previews,
tables, distributions and correlations. Complete statistics for all 321 clients
are exported to CSV only. Models use only lags of the target client.

In [ ]:
DATA_DIR = ROOT / "data/electricity_notebook"
df = load_electricity(DATA_DIR)
assert isinstance(TARGET_COLUMN, int) and not isinstance(TARGET_COLUMN, bool)
assert 0 <= TARGET_COLUMN < df.shape[1]
TARGET = df.columns[TARGET_COLUMN]
y = df[TARGET].to_numpy(dtype=float)
assert np.isfinite(y).all(), "Target must be finite; no automatic imputation."
# Keep client displays strictly limited to the first five source columns.
DISPLAY_COLUMNS = list(df.columns[:5])
print("Source:", SOURCE_URL)
print("SHA-256:", SOURCE_SHA256)
print(f"Target: {TARGET}; {len(y):,} observations; original numeric scale.")
display(df.loc[:, DISPLAY_COLUMNS].head())

### I.1) Dimensions, data quality and descriptive statistics — first five clients

In [ ]:
quality, descriptive = describe_data(df)
display_descriptive = descriptive.loc[DISPLAY_COLUMNS]
display(quality.to_frame())
print("Statistics cover all 321 clients; only the first five clients are displayed below.")
print("Complete per-client statistics are exported to CSV, not displayed.")
with pd.option_context("display.max_rows", 5, "display.max_columns", None):
    display(display_descriptive)
if WRITE_RESULTS:
    quality.to_csv(OUTPUT / "data_quality.csv")
    descriptive.to_csv(OUTPUT / "descriptive_all_clients.csv")

### I.2) Target trajectory and distributions of the first five clients

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 7), layout="constrained")
axes[0].plot(df.index, y, color=OVENTI["light"], lw=0.65)
axes[0].set(title=f"OVENTI | {TARGET} — complete trajectory", ylabel="Published load scale")
zoom = df.iloc[-14 * 24 :]
axes[1].plot(zoom.index, zoom[TARGET], color=OVENTI["teal"], lw=1)
axes[1].set(title="Descriptive zoom — final 336 observations", ylabel="Published load scale")
for ax in axes:
    ax.set_xlabel("Observation index (hourly cadence; no source dates)")
reporter.save(fig, "01_trajectory")
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(13, 6), layout="constrained")
for ax, name in zip(axes.flat, DISPLAY_COLUMNS, strict=False):
    sns.histplot(
        df[name], bins=45, ax=ax, color=OVENTI["orange"] if name == TARGET else OVENTI["light"]
    )
    ax.axvline(df[name].mean(), color=OVENTI["dark"], ls="--", label="Mean")
    ax.axvline(df[name].median(), color=OVENTI["teal"], ls=":", label="Median")
    ax.set(title=name, xlabel="Published load scale", ylabel="Observations")
    ax.legend(fontsize=8)
for ax in axes.flat[len(DISPLAY_COLUMNS) :]:
    ax.axis("off")
fig.suptitle("OVENTI | First five clients — marginal distributions, no standardization")
reporter.save(fig, "02_client_histograms")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8), layout="constrained")
sns.histplot(np.diff(y), bins=70, ax=axes[0, 0], color=OVENTI["teal"])
axes[0, 0].set(title=f"{TARGET} — hourly increments", xlabel="Y[t] − Y[t−1]")
sns.histplot(display_descriptive["mean"], bins=5, ax=axes[0, 1], color=OVENTI["light"])
axes[0, 1].set(
    title="First five clients — distribution of client means", xlabel="Mean published load"
)
sns.histplot(display_descriptive.zero_fraction, bins=5, ax=axes[1, 0], color=OVENTI["orange"])
axes[1, 0].set(title="First five clients — fraction of zero values", xlabel="Zero fraction")
axes[1, 1].scatter(
    display_descriptive["mean"],
    display_descriptive.std_population,
    s=30,
    alpha=0.8,
    color=OVENTI["light"],
)
axes[1, 1].set(
    title="First five clients — mean versus variability",
    xlabel="Client mean",
    ylabel="Population standard deviation",
)
reporter.save(fig, "03_distribution_diagnostics")
plt.show()

fig, ax = plt.subplots(figsize=(8, 6), layout="constrained")
sns.heatmap(
    df[DISPLAY_COLUMNS].corr(),
    annot=True,
    fmt=".2f",
    vmin=-1,
    vmax=1,
    cmap=LinearSegmentedColormap.from_list(
        "oventi", [OVENTI["orange"], "#FFFFFF", OVENTI["light"]]
    ),
    square=True,
    ax=ax,
)
ax.set_title("OVENTI | First five clients — contemporaneous correlations")
reporter.save(fig, "04_correlations")
plt.show()

## II) Forecasting and rolling backtesting

### Target, information set and chronology

At origin $t-1$, the available information is $\mathcal F_{t-1}=\sigma(Y_0,\ldots,Y_{t-1})$.
We forecast $\widehat Y_t$ and issue $[L_t,U_t]$ before revealing $Y_t$.
Regression models use $(Y_{t-1},\ldots,Y_{t-\ell})$, with
$\ell=\texttt{n\_lags}$. No future covariates or calendar features are used.

**Train → sequential calibration → sequential test**, then move to the next window.
Test blocks are adjacent and disjoint; historical windows overlap and remain dependent.
A previous test target can legitimately become an observed past value in a later fold.
`rolling_forecasts` generates all forecasts; `replay` enforces calibration chronology.
A Calibre origin is the index of the **last known observation**: the target is at `origin + 1`.

`Panel` requires a regular calendar: a technical hourly index is created internally,
**without presenting it as source dates** or using it as a feature.

In [ ]:
SPLITS = chronological_splits(len(y), CONFIG)
split_table = pd.DataFrame(
    [
        {
            "fold": s.fold,
            "train_from": s.train_start,
            "train_to": s.train_end - 1,
            "calibration_from": s.train_end,
            "calibration_to": s.test_start - 1,
            "test_from": s.test_start,
            "test_to": s.test_end - 1,
            "n_train": s.train_end - s.train_start,
            "n_calibration": s.test_start - s.train_end,
            "H": s.test_end - s.test_start,
        }
        for s in SPLITS
    ]
)
display(split_table)
print(
    f"Forecast lead h=1; H={CONFIG.test_size} test targets/fold; "
    f"{CONFIG.n_split * CONFIG.test_size} evaluated targets/pair."
)

fig, ax = plt.subplots(figsize=(15, max(4, CONFIG.n_split * 0.6)), layout="constrained")
for s in SPLITS:
    for left, right, label, color in [
        (s.train_start, s.train_end, "Train", OVENTI["teal"]),
        (s.train_end, s.test_start, "Calibration", OVENTI["orange"]),
        (s.test_start, s.test_end, "Test", OVENTI["light"]),
    ]:
        ax.barh(
            s.fold,
            right - left,
            left=left,
            height=0.65,
            color=color,
            label=label if s.fold == 0 else None,
        )
ax.set(
    yticks=range(CONFIG.n_split),
    xlabel="Observation index",
    ylabel="Fold",
    title="OVENTI | Rolling chronology — train → calibration → test",
)
ax.invert_yaxis()
ax.legend(loc="upper left")
reporter.save(fig, "05_rolling_backtest")
plt.show()

### II.1) ACF, PACF and SARIMA identification — training only

For a stationary series $Z_t$, $\rho(k)=\operatorname{Corr}(Z_t,Z_{t-k})$ is
the ACF; the PACF is the partial correlation after controlling for intermediate lags.
A PACF cutoff suggests an AR($p$) term, while an ACF cutoff suggests an MA($q$) term;
peaks at multiples of $s=24$ suggest a seasonal component.
These heuristics do not determine an order on their own; the displayed bands
do not adjust for multiple comparisons.

We fix $d=1$, $(P,D,Q)=(1,0,0)$ and $s=24$, then compare $p,q\in\{0,1,2\}$
using **AIC on the first training block only**. The selected order is shared by
subsequent folds, without test-set optimization. AICc and BIC are also reported.
Seasonal differencing is shown as a diagnostic, not imposed on the model.

In [ ]:
first = SPLITS[0]
training = y[first.train_start : first.train_end]
first_difference = np.diff(training, n=CONFIG.sarima_d)
seasonal_difference = first_difference[CONFIG.period :] - first_difference[: -CONFIG.period]
fig, axes = plt.subplots(3, 2, figsize=(15, 11), layout="constrained")
for row, (label, z) in enumerate(
    [
        ("Raw training target", training),
        (f"Differenced training target (d={CONFIG.sarima_d})", first_difference),
        ("Additional seasonal difference (diagnostic only)", seasonal_difference),
    ]
):
    lags = min(4 * CONFIG.period, len(z) // 2 - 1)
    plot_acf(z, lags=lags, zero=False, ax=axes[row, 0], color=OVENTI["light"])
    plot_pacf(z, lags=lags, zero=False, method="ywm", ax=axes[row, 1], color=OVENTI["teal"])
    axes[row, 0].set_title(f"{label} — ACF")
    axes[row, 1].set_title(f"{label} — PACF")
    for ax in axes[row]:
        for k in range(CONFIG.period, lags + 1, CONFIG.period):
            ax.axvline(k, color=OVENTI["orange"], alpha=0.35, ls=":")
reporter.save(fig, "06_training_acf_pacf")
plt.show()

SARIMA_ORDER, sarima_identification = identify_sarima(
    training,
    CONFIG,
    max_p=SARIMA_MAX_P,
    max_q=SARIMA_MAX_Q,
)
display(sarima_identification)
print(
    "Training-AIC selected order:",
    SARIMA_ORDER,
    "seasonal:",
    (*CONFIG.seasonal_order, CONFIG.period),
)

### II.2) Repository models and update policy

| Forecaster | Calibre API + associated package | Across forecast origins |
|---|---|---|
| Moving average | `StatsForecastModel(WindowAverage)` | Mean of the last 24 revealed values |
| Trend–season decomposition | `StatsForecastModel(MSTL)` | Past-only decomposition on bounded history; ETS(A,A,N) trend and repeated seasonality |
| SARIMA | `StatsForecastModel(ARIMA)` | Re-estimation on bounded past history at every origin; native Gaussian PI |
| CatBoost | `MLForecast(CatBoostRegressor)` | Trees fitted on training data, then frozen; newly observed lags |
| OGD | `MLForecast(SGDRegressor)` | One chronological pass without shuffling, replayed on each expanding prefix |

The local `VendorWindow` wrapper only limits history length and retains native
SARIMA bounds: **no new loop over forecast origins is introduced**.
The repository's StatsForecast adapter actually refits local models on every
`predict` call, even with `refit_every=None`. Retained history is bounded by
`train_size` for SARIMA and MSTL; there is no fixed-parameter SARIMA filter.

For OGD, the centering and scale of $Y$ are computed on the **training data only**
and remain fixed. With $x_t=(z_{t-1},\ldots,z_{t-\ell})$, SGD uses the loss
$\frac12(z_t-w^\top x_t-b)^2+\frac\lambda2\|w\|_2^2$ and a constant step size $\eta$.
With `max_iter=1`, `shuffle=False`, `tol=None` and this fixed scale, replaying each
prefix from the same initial coefficients reproduces sequential OGD updates.
This is exact but more expensive than a persistent `partial_fit`,
which the Calibre API does not provide. No update uses the target being forecast.
Point forecasts and intervals are then returned to the original scale.

### Forecast metrics and uncertainty in mean performance

For the $H$ targets in a test block, define $e_t=\widehat Y_t-Y_t^{\mathrm{true}}$:

$$RMSE=\sqrt{H^{-1}\sum_t e_t^2},\qquad
MAE=H^{-1}\sum_t|e_t|,\qquad \mathrm{bias}=H^{-1}\sum_t e_t,$$

$$\widehat{\mathrm{coverage}}=H^{-1}\sum_t\mathbf1\{L_t\le Y_t\le U_t\},\qquad
\overline{\mathrm{width}}=H^{-1}\sum_t(U_t-L_t).$$

Here $H$ is the **test-block length**, not the forecast lead $h=1$.
The final bars show mean fold metrics (therefore the mean fold RMSE,
not the square root of the pooled MSE).

ACI/PID can produce empty or unbounded sets. **All test dates are retained**,
with coverage 0 for an empty set, length 0 for an empty set and $+\infty$ for
an unbounded set. The formula $U_t-L_t$ applies to ordered intervals;
this length convention specifies its extension.
Counts of empty/unbounded sets are always reported, never filtered out.

## III) Conformal calibration for time series

For $S_t=|Y_t-\widehat Y_t|$, ACI/PID issue
$[\widehat Y_t-q_t,\widehat Y_t+q_t]$.

- **EnbPI (residual-only)**: `EnbPI` + `Asymmetric`, with a rolling memory of
  signed residuals $R_t=Y_t-\widehat Y_t$; the interval is
  $[\widehat Y_t+Q(\beta),\widehat Y_t+Q(1-\alpha+\beta)]$, searching for
  the shortest empirical quantile pair. Initialization uses **chronological
  calibration forecasts**, not in-sample errors.
  **This is not the complete Xu–Xie bootstrap/OOB EnbPI algorithm.** The repository
  separates `BootstrapEnsemble` (frozen lag regression) from the `EnbPI` calibrator.
  To compare the same four forecasters, we apply only this sequential
  calibrator, including to re-estimated/OGD centers; no guarantee specific to
  the OOB algorithm is claimed.
- **ACI**: `ACI(SplitQuantile(...))`, using the $n+1$-corrected conformal rank
  on revealed scores; level adaptation is $\ell_{t+1}=\ell_t+\gamma(m_t-\alpha)$,
  where $m_t$ indicates a miss of the threshold actually issued. Calibration
  is replayed before testing, without accessing future targets.
- **Conformal PID**: the repository's `ConformalPID`, with
  its built-in causal AR(1) scorecasting to anticipate scores. The controller is explicitly
  initialized with calibration scores known at the start of testing, then updated
  using only revealed feedback. Gains use the fixed seed scale, as specified in
  `docs/semantics.md`.
- **SARIMA**: no conformal calibration. The native Gaussian PI is
  $\widehat Y_t\pm z_{1-\alpha/2}\widehat\sigma_{t\mid t-1}$, conditional on
  the fitted model; it is not a CI for the mean, and parameter estimation
  uncertainty is not fully propagated.

`EnbPI`, `ACI` and `ConformalPID` are imported from the public `calibre` API.
Their implementations live in `src/calibre/conformal/calibrators/`.
No conditional, simultaneous or distribution-free coverage under arbitrary
dependence is promised. The same point forecasts are reused for a forecaster's
three conformal methods, so the comparison isolates calibration.

### III.1) Moving average — three calibrators

In [ ]:
moving_results = run_family("Moving average", y, SPLITS, CONFIG)
display(fold_metrics(moving_results))
reporter.plot_pairs(moving_results)

### III.2) MSTL trend–season decomposition — three calibrators

In [ ]:
decomposition_results = run_family("Trend-season MSTL", y, SPLITS, CONFIG)
display(fold_metrics(decomposition_results))
reporter.plot_pairs(decomposition_results)

### III.3) SARIMA — Gaussian interval from the literature, no conformal calibration

In [ ]:
sarima_results = run_family("SARIMA", y, SPLITS, CONFIG, sarima_order=SARIMA_ORDER)
display(fold_metrics(sarima_results))
reporter.plot_pairs(sarima_results)

### III.4) CatBoost — three calibrators

In [ ]:
catboost_results = run_family("CatBoost", y, SPLITS, CONFIG)
display(fold_metrics(catboost_results))
reporter.plot_pairs(catboost_results)

### III.5) Online Gradient Descent — three calibrators

In [ ]:
ogd_results = run_family("Online GD", y, SPLITS, CONFIG)
display(fold_metrics(ogd_results))
reporter.plot_pairs(ogd_results)

### III.6) Complete results table and CIs for mean performance

For each pair, the $K$ fold metrics $m_1,\ldots,m_K$ give
$\bar m=K^{-1}\sum_k m_k$. We resample circular, contiguous blocks of two folds,
using **the same bootstrap indices for every pair**.
The 2.5% and 97.5% quantiles of the resampled means give 95% percentile CIs.
Because folds are dependent and few in number, these CIs remain **exploratory**;
they are neither PIs nor an exact coverage guarantee.
Infinite lengths remain infinite in both the results table and the bootstrap.

**One final figure per forecaster** contains all five metric bar charts, with
only that forecaster's interval methods on the x-axis (at most three bars per chart;
one native Gaussian interval for SARIMA). The complete results table stays unchanged.

In [ ]:
all_predictions = pd.concat(
    [
        moving_results,
        decomposition_results,
        sarima_results,
        catboost_results,
        ogd_results,
    ],
    ignore_index=True,
)
all_fold_metrics = fold_metrics(all_predictions)
all_summary = summarize(
    all_fold_metrics,
    confidence=CONFIDENCE_METRICS,
    n_bootstrap=N_BOOTSTRAP_CI,
    block_length=BOOTSTRAP_BLOCK_LENGTH,
    seed=CONFIG.seed,
)
# Audits of completeness and equality of evaluated targets; no method-specific filtering.
assert len(all_summary) == 13
assert (all_summary.n_targets == CONFIG.n_split * CONFIG.test_size).all()
assert all_predictions.groupby(["family", "method", "fold"]).size().eq(CONFIG.test_size).all()
for _, group in all_predictions.groupby(["family", "fold"]):
    assert group.groupby("step").point.nunique().eq(1).all()
assert not all_predictions[["truth", "point", "lower", "upper"]].isna().any().any()
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(all_summary)

In [ ]:
# One five-metric figure per forecaster; all result tables remain complete.
# Apply the RMSE threshold to these figures only, never to tables or exports.
PLOT_RMSE_LIMIT = 1e6
_ = reporter.plot_comparison(all_summary, rmse_limit=PLOT_RMSE_LIMIT, auto_symlog=True)

### III.7) Exports and traceability

In [ ]:
versions = {
    package: importlib.metadata.version(package)
    for package in [
        "libcalibre",
        "numpy",
        "pandas",
        "statsforecast",
        "mlforecast",
        "catboost",
        "scikit-learn",
        "statsmodels",
        "matplotlib",
        "nbformat",
    ]
}
manifest = {
    "created_at_utc": CREATED_AT.isoformat(),
    "python": platform.python_version(),
    "local_calibre": str(Path(calibre.__file__).resolve()),
    "versions": versions,
    "source_url": SOURCE_URL,
    "source_sha256": SOURCE_SHA256,
    "target": TARGET,
    "display_clients": DISPLAY_COLUMNS,
    "comparison_layout": "One five-metric figure per forecaster",
    "configuration": asdict(CONFIG),
    "sarima_order_training_aic": SARIMA_ORDER,
    "sarima_max_p": SARIMA_MAX_P,
    "sarima_max_q": SARIMA_MAX_Q,
    "metric_ci_confidence": CONFIDENCE_METRICS,
    "metric_ci_replicates": N_BOOTSTRAP_CI,
    "metric_ci_block_length_folds": BOOTSTRAP_BLOCK_LENGTH,
    "plot_fold": PLOT_FOLD,
    "plot_rmse_limit_only": PLOT_RMSE_LIMIT,
    "enbpi_scope": "Sequential residual calibrator only; not bootstrap/OOB EnbPI",
    "ogd_scope": "Exact deterministic one-pass prefix replay; fixed train scaling",
    "empty_set_width": 0,
    "unbounded_set_width": "+infinity",
}
if WRITE_RESULTS:
    all_predictions.to_csv(OUTPUT / "predictions_all_pairs.csv", index=False)
    all_fold_metrics.to_csv(OUTPUT / "metrics_by_fold.csv", index=False)
    all_summary.to_csv(OUTPUT / "summary_with_mean_metric_ci.csv", index=False)
    split_table.to_csv(OUTPUT / "chronological_splits.csv", index=False)
    sarima_identification.to_csv(OUTPUT / "sarima_training_identification.csv", index=False)
    (OUTPUT / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print("Complete tables, manifest and figures:", OUTPUT)
display(pd.Series(versions, name="Version"))

### Selected references

- Contracts and chronology: [`architecture.md`](../docs/architecture.md),
  [`semantics.md`](../docs/semantics.md).
- [Lai et al., LSTNet (2018)](https://arxiv.org/abs/1703.07015) — ECL matrix;
  do not confuse it with the raw 15-minute UCI file.
- [Hyndman & Athanasopoulos, *Forecasting: Principles and Practice*, ARIMA](https://otexts.com/fpp3/arima.html).
- [Bandara et al., MSTL](https://arxiv.org/abs/2107.13462).
- [Xu & Xie (2021), EnbPI](https://proceedings.mlr.press/v139/xu21h.html) —
  this experiment reproduces only its residual calibration component.
- [Gibbs & Candès (2021), ACI](https://arxiv.org/abs/2106.00170).
- [Angelopoulos, Candès & Tibshirani, Conformal PID](https://arxiv.org/abs/2307.16895).